# LSTM, GRU: text classification


In [ ]:
import os
import re
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import wandb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

from helpers.config import DATA_DIR, PROCESSED_DIR

DATA_PATH = DATA_DIR / "nlp-getting-started" / "train.csv"
OUT_DIR = PROCESSED_DIR / "lecture" / "lstm_gru_classification"
assert DATA_PATH.exists(), f"{DATA_PATH} missing: download via kaggle CLI"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Data


In [ ]:
df_twitter = pd.read_csv(DATA_PATH)
df_twitter.sample(5)

In [ ]:
# straightforward train/test split

X_train, X_test, y_train, y_test = train_test_split(
    df_twitter.text,
    df_twitter.target,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=df_twitter.target,
)

## Tokenizers


In [ ]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1
WORD_VOCAB_MAX = 20_000  # cap


def word_tokenize(text: str):
    # Lowercasing and basic tokenization: split on non-word characters
    text = text.lower()
    toks = re.findall(r"[a-z0-9_]+", text)
    return toks


word_counts = Counter()
for t in df_twitter.text.tolist():
    word_counts.update(word_tokenize(t))

print(f"Total unique words: {len(word_counts)}")

# Build vocab: [<pad>, <unk>] + most common words
most_common = [w for w, _ in word_counts.most_common(WORD_VOCAB_MAX - 2)]
word_vocab = [PAD_TOKEN, UNK_TOKEN] + most_common
word2id = {w: i for i, w in enumerate(word_vocab)}
id2word = {i: w for w, i in word2id.items()}


def encode_words(text: str):
    toks = word_tokenize(text)
    return [word2id.get(tok, UNK_ID) for tok in toks]


vocab_size_word = len(word2id)
vocab_size_word

In [ ]:
# Prepare iterator over training texts
def text_iterator(series, batch_size=1000):
    batch = []
    for text in series:
        batch.append(text.lower())
        if len(batch) >= batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


BPE_VOCAB_SIZE = 1_000
bpe_tokenizer = Tokenizer(models.BPE(unk_token=UNK_TOKEN))
bpe_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(
    vocab_size=BPE_VOCAB_SIZE, special_tokens=[PAD_TOKEN, UNK_TOKEN]
)
bpe_tokenizer.train_from_iterator(text_iterator(df_twitter["text"]), trainer=trainer)

# Make PAD_ID=0 across both tokenizers for convenience by remapping if necessary
# (tokenizers library stores vocab as token->id dict; we will just keep PAD as a special pad token externally)
bpe_vocab = bpe_tokenizer.get_vocab()
len(bpe_vocab)

In [ ]:
# Helper wrappers for BPE encode/decode to IDs with PAD=0 fallback
# We'll store PAD as 0 and UNK as the tokenizer's unk id for consistency.
BPE_PAD_ID = 0
BPE_UNK_ID = bpe_tokenizer.token_to_id(UNK_TOKEN)


def encode_bpe(text: str):
    ids = bpe_tokenizer.encode(text.lower()).ids
    # Keep original ids; padding will use 0 in collator
    return ids


# Quick smoke test
print(bpe_tokenizer.encode("fire at the refinery").tokens)
print(encode_bpe("fire at the refinery")[:10])

## Datasets


In [ ]:
class SeqDataset(Dataset):
    def __init__(self, texts, labels, encoder: str):
        self.texts = list(texts)
        self.labels = list(labels)
        self.encoder = encoder  # 'word' or 'bpe'

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        t = self.texts[idx]
        y = int(self.labels[idx])
        if self.encoder == "word":
            ids = encode_words(str(t).lower().strip())
        else:
            ids = encode_bpe(str(t).lower().strip())
        return torch.tensor(ids, dtype=torch.long), torch.tensor(y, dtype=torch.long)


train_word_ds = SeqDataset(X_train.tolist(), y_train, "word")
val_word_ds = SeqDataset(X_test.tolist(), y_test, "word")
train_bpe_ds = SeqDataset(X_train.tolist(), y_train, "bpe")
val_bpe_ds = SeqDataset(X_test.tolist(), y_test, "bpe")
len(train_word_ds), len(train_bpe_ds), len(val_word_ds), len(val_bpe_ds)

In [ ]:
def make_collate_fn(pad_id: int):
    def collate(batch):
        seqs, labels = zip(*batch)
        lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
        padded = pad_sequence(seqs, batch_first=True, padding_value=pad_id)
        return {
            "input_ids": padded,
            "lengths": lengths,
            "labels": torch.tensor(labels, dtype=torch.long),
        }

    return collate


word_collate = make_collate_fn(PAD_ID)  # 0
bpe_collate = make_collate_fn(BPE_PAD_ID)  # 0

## Model


In [ ]:
class UniversalRNNClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_rnn_layers: int = 3,
        embed_dim: int = 256,
        hidden_dim: int = 256,
        num_classes: int = 2,
        pad_id: int = 0,
        rnn_type=nn.RNN,
        bidirectional: bool = False,
    ):
        super().__init__()
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        self.num_layers = num_rnn_layers
        self.hidden_dim = hidden_dim
        self.rnn_type = rnn_type

        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.rnn = rnn_type(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            num_layers=num_rnn_layers,
            bidirectional=bidirectional,
        )
        self.fc = nn.Linear(hidden_dim * self.num_directions, num_classes)

    def forward(self, input_ids=None, lengths=None, labels=None, **kwargs):
        # input_ids: [B, T], lengths: [B]
        emb = self.embed(input_ids)  # [B, T, E]

        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        rnn_out = self.rnn(packed)

        # Unify hidden extraction across RNN/GRU/LSTM
        if isinstance(rnn_out[1], tuple):  # LSTM case
            h_n, _ = rnn_out[1]
        else:  # RNN / GRU
            h_n = rnn_out[1]

        # h_n shape: [num_layers * num_directions, B, H]
        if self.bidirectional:
            last_layer_f = h_n[-2, :, :]
            last_layer_b = h_n[-1, :, :]
            last_h = torch.cat([last_layer_f, last_layer_b], dim=-1)
        else:
            last_h = h_n[-1, :, :]

        logits = self.fc(last_h)  # [B, C]

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits[:, 1]
    auc = roc_auc_score(labels, preds)
    return {"auc": auc}

## Gradient flow probe


In [ ]:
CELL_TYPES = {nn.RNN: nn.RNNCell, nn.GRU: nn.GRUCell, nn.LSTM: nn.LSTMCell}


# ---- Copy trained nn.RNN / nn.GRU / nn.LSTM weights into a stack of cells (one per layer) ----
def build_cells_from_module(rnn_module):
    cells = nn.ModuleList()
    for l in range(rnn_module.num_layers):
        in_size = rnn_module.input_size if l == 0 else rnn_module.hidden_size
        extra = (
            {"nonlinearity": rnn_module.nonlinearity}
            if type(rnn_module) is nn.RNN
            else {}
        )
        cell = CELL_TYPES[type(rnn_module)](
            in_size, rnn_module.hidden_size, bias=rnn_module.bias, **extra
        )
        # weight_ih_l{l}, weight_hh_l{l}, bias_ih_l{l}, bias_hh_l{l}
        cell.weight_ih.data.copy_(getattr(rnn_module, f"weight_ih_l{l}").data)
        cell.weight_hh.data.copy_(getattr(rnn_module, f"weight_hh_l{l}").data)
        if rnn_module.bias:
            cell.bias_ih.data.copy_(getattr(rnn_module, f"bias_ih_l{l}").data)
            cell.bias_hh.data.copy_(getattr(rnn_module, f"bias_hh_l{l}").data)
        cells.append(cell)
    return cells


@torch.no_grad()
def _random_embeds(model, T=120, B=1, device="cpu"):
    ids = torch.randint(0, model.embed.num_embeddings, (B, T), device=device)
    return model.embed(ids)  # [B, T, E]


# ---- Step-by-step unroll of copied cells; return top-layer states: {"h": [...]}, plus "c" for LSTM ----
def unroll_cells_with_grad_seed(model, T=120, device="cpu"):
    model.eval()
    model.to(device)
    is_lstm = isinstance(model.rnn, nn.LSTM)

    cells = build_cells_from_module(model.rnn).to(device)
    x = _random_embeds(model, T=T, B=1, device=device)  # inputs don't need grads

    H, L = model.hidden_dim, model.num_layers
    # Seed states with requires_grad=True so the recurrence is differentiable wrt h_0/c_0
    hs = [torch.zeros(1, H, device=device, requires_grad=True) for _ in range(L)]
    cs = [torch.zeros(1, H, device=device, requires_grad=True) for _ in range(L)]

    h_top_list, c_top_list = [], []
    for t in range(T):
        inp = x[:, t, :]  # [1, E]
        for l in range(L):
            if is_lstm:
                hs[l], cs[l] = cells[l](inp, (hs[l], cs[l]))
            else:
                hs[l] = cells[l](inp, hs[l])
            inp = hs[l]
        h_top_list.append(hs[-1])  # [1, H], requires_grad=True
        c_top_list.append(cs[-1])  # [1, H], requires_grad=True
    return {"h": h_top_list, "c": c_top_list} if is_lstm else {"h": h_top_list}


# ---- Jacobian-proxy norms for a sequence of states s_list (e.g., h_t or c_t) ----
def jacobian_proxy_norms(s_list, max_lag=60):
    """
    Proxy: phi(s_{t+s}) = 1/2 ||s_{t+s}||^2; grad wrt s_t gives (∂s_{t+s}/∂s_t)^T s_{t+s}.
    Returns lags [S], avg_norms [S], heat [T, S+1].
    """
    T = len(s_list)
    if T < 2:
        return torch.tensor([]), torch.tensor([]), torch.zeros(0, 0)
    S = min(max_lag, T - 1)
    device = s_list[0].device
    heat = torch.zeros(T, S + 1, device=device)
    lags, norms = [], []
    for s in range(1, S + 1):
        vals = []
        for t in range(0, T - s):
            phi = 0.5 * (s_list[t + s] ** 2).sum()
            (g_t,) = torch.autograd.grad(
                phi, (s_list[t],), retain_graph=True, allow_unused=True
            )
            if g_t is None:
                continue
            v = g_t.norm().item()
            vals.append(v)
            heat[t, s] = v
        if vals:
            lags.append(s)
            norms.append(sum(vals) / len(vals))
    return torch.tensor(lags), torch.tensor(norms), heat[:, : S + 1].detach().cpu()


def half_life(lags, norms, frac=0.5):
    if len(lags) == 0:
        return None
    base = norms[0].item()
    thr = frac * base
    for s, v in zip(lags.tolist(), norms.tolist()):
        if v <= thr:
            return s
    return None


def plot_curve(lags, norms, label, out_path=None, title=None):
    plt.figure()
    plt.plot(lags, norms, label=label)
    plt.xlabel("Lag s")
    plt.ylabel("Average ||∂·_{t+s} / ∂·_t||_2 (proxy)")
    plt.title(title or label)
    plt.legend()
    if out_path:
        plt.savefig(out_path, bbox_inches="tight")
    plt.show()


def plot_heatmap(heat, out_path=None, title=None):
    plt.figure()
    plt.imshow(heat.numpy(), aspect="auto", origin="lower")
    plt.colorbar(label="||∂·_{t+s} / ∂·_t||_2 (proxy)")
    plt.xlabel("lag s")
    plt.ylabel("time t")
    if title:
        plt.title(title)
    if out_path:
        plt.savefig(out_path, bbox_inches="tight")
    plt.show()

## Train RNN, GRU, LSTM


In [ ]:
BATCH_TRAIN = 64
BATCH_EVAL = 64
EPOCHS = 10
LR = 2e-4
PROBE_T = 120
MAX_LAG = 60

results = []
curves = {}

for rnn_type in [nn.RNN, nn.GRU, nn.LSTM]:
    name = rnn_type.__name__
    # Build model
    word_model = UniversalRNNClassifier(
        vocab_size=vocab_size_word, pad_id=PAD_ID, rnn_type=rnn_type
    )

    wandb.init(
        project="rnn-lecture-classification", name=f"word-{name.lower()}", dir=OUT_DIR
    )

    # HF Trainer args
    word_args = TrainingArguments(
        output_dir=str(OUT_DIR / f"out_word_{name.lower()}"),
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,
        report_to=["wandb"],
        run_name=f"word-{name.lower()}",
        load_best_model_at_end=True,
        metric_for_best_model="auc",
        greater_is_better=True,
        seed=42,
        dataloader_pin_memory=False,  # not supported on MPS
    )

    trainer_word = Trainer(
        model=word_model,
        args=word_args,
        train_dataset=train_word_ds,
        eval_dataset=val_word_ds,
        data_collator=word_collate,
        compute_metrics=compute_metrics,
    )

    print(name, "word-level model params:", count_params(word_model))
    trainer_word.train()
    best_auc = trainer_word.evaluate().get("eval_auc", float("nan"))
    wandb.finish()

    # Unroll with cell stack (weights copied) and grad-seeded initial states
    states = unroll_cells_with_grad_seed(word_model, T=PROBE_T, device="cpu")
    for state, s_list in states.items():
        label = f"{name} ({state})"
        base = f"{name.lower()}_{state}"
        lags, norms, heat = jacobian_proxy_norms(s_list, max_lag=MAX_LAG)
        curves[label] = (lags, norms)

        plot_curve(
            lags,
            norms,
            label,
            out_path=OUT_DIR / f"{base}_curve.png",
            title=f"{label}: gradient vs lag",
        )
        plot_heatmap(
            heat,
            out_path=OUT_DIR / f"{base}_heat.png",
            title=f"{label}: t×lag map",
        )

        results.append(
            {
                "model_name": name,
                "state": state,
                "num_params": count_params(word_model),
                "best_auc": best_auc,
                "grad_half_life_50pct_lag": half_life(lags, norms, 0.5),
                "grad_half_life_10pct_lag": half_life(lags, norms, 0.1),
            }
        )

df_summary = pd.DataFrame(results)
df_summary.to_csv(OUT_DIR / "summary_grad_flow.csv", index=False)
df_summary

In [ ]:
plt.figure()
for label, (lags, norms) in curves.items():
    plt.plot(lags, norms, label=label)
plt.xlabel("Lag s")
plt.ylabel("Average ||∂·_{t+s} / ∂·_t||_2 (proxy)")
plt.title("Gradient-through-time decay: RNN vs GRU vs LSTM (h & c)")
plt.legend()
plt.savefig(OUT_DIR / "combined_grad_curves.png", bbox_inches="tight")
plt.show()